# Sample 1 — where do the errors come from?

Two suspects:

1. **the reader** (pdfplumber) — did it damage the document?
2. **the models** — did they label it wrong?

Three steps, one verdict.


In [1]:
import os
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

import json
import pandas as pd
from IPython.display import display

from dmpbridge.core import paths as P
from dmpbridge.evaluation.evaluate import (
    containment, extract_gold, resolve_old_gt_path, tokenize,
)

SAMPLE, EXTRACTOR = 1, 'pdfplumber'
MODELS = ['llama3.1:8b', 'gemma4:e4b', 'llama3.3:70b']
pd.set_option('display.max_colwidth', 80)

# The human annotation: the correct answer for this document.
gold = extract_gold(resolve_old_gt_path(SAMPLE))
# What pdfplumber read from the PDF.
blocks = json.loads((P.EXTRACTED_DIR / EXTRACTOR / f'sample{SAMPLE}.json')
                    .read_text(encoding='utf-8'))
# What each model labeled — the same blocks, one label per block.
labeled = {m: json.loads(P.labeled_path(P.make_tag(m, EXTRACTOR), SAMPLE)
                         .read_text(encoding='utf-8')) for m in MODELS}


## Step 1 — Did the reader damage the document?

The human annotation says what is really in this document. We check every block the
reader produced: does its text belong to something in the annotation?


In [2]:
# For each block, find the annotation item its text belongs to.
truth = []
for b in blocks:
    bt = tokenize(b['text'])
    best, lab = 0.0, None
    for gt, gl in gold:
        c = containment(bt, tokenize(gt))
        if c > best:
            best, lab = c, gl
    truth.append(lab if best >= 0.75 else None)

lost = sum(1 for t in truth if t is None)
print(f'the annotation has {len(gold)} items')
print(f'the reader produced {len(blocks)} blocks')
print(f'blocks with text that is NOT in the annotation: {lost}')


the annotation has 28 items
the reader produced 78 blocks
blocks with text that is NOT in the annotation: 0


**Verdict on the reader: innocent.** Every block is real text from the document.
Nothing was lost, nothing was made up.

The reader did one thing worth knowing: it reads **line by line**, so a long answer
arrives as several blocks instead of one. That is why there are 78 blocks for only
28 items. Splitting is not an error by itself — the pipeline glues neighbouring
blocks back together when they carry the same label.


## Step 2 — How well did each model label the same blocks?

Every model received the identical blocks. Each block's correct label is known
(from the annotation). So this table is a pure test of the models:


In [3]:
known = sum(1 for t in truth if t)
rows = []
for m in MODELS:
    ok = sum(1 for t, b in zip(truth, labeled[m]) if t and b.get('label') == t)
    rows.append({'model': m, 'blocks': known, 'correct': ok,
                 'wrong': known - ok, 'accuracy': ok / known})
acc = pd.DataFrame(rows).set_index('model')
display(acc.style.background_gradient(cmap='Greens', subset=['accuracy'])
        .format({'accuracy': '{:.0%}'}))


,blocks,correct,wrong,accuracy
model,,,,
llama3.1:8b,78,67,11,86%
gemma4:e4b,78,78,0,100%
llama3.3:70b,78,77,1,99%


One model labels **every block correctly** — from exactly the same input the others
got. That settles it: the input was good enough to score 100%. Any error below is
the model's.


## Step 3 — The wrong blocks, one by one


In [4]:
for m in MODELS:
    bad = [{'true label': t, 'model said': b.get('label'), 'text': b['text']}
           for t, b in zip(truth, labeled[m]) if t and b.get('label') != t]
    print(f'{m}: {len(bad)} wrong')
    if bad:
        display(pd.DataFrame(bad))
    print()


llama3.1:8b: 11 wrong


,true label,model said,text
0,question.text,section.title,"B. Scientific data that will be preserved and shared, and the rationale for ..."
1,answer.text,section.description,about other studies.
2,question.text,section.title,"C. Metadata, other relevant data, and associated documentation:"
3,answer.text,question.text,The following data will be created as a result of this project:
4,question.text,section.title,A. Repository where scientific data and metadata will be archived:
5,question.text,section.title,B. How scientific data will be findable and identifiable:
6,question.text,section.title,C. When and how long the scientific data will be made available:
7,question.text,section.title,"A. Factors affecting subsequent access, distribution, or reuse of scientific..."
8,answer.text,question.text,about using their data:
9,question.text,section.title,B. Whether access to scientific data will be controlled:



gemma4:e4b: 0 wrong

llama3.3:70b: 1 wrong


,true label,model said,text
0,answer.text,section.description,The following data will be created as a result of this project:


## Conclusion

**The errors come from the model, not from pdfplumber.**

The proof is simple: gemma4 received exactly the same blocks and labeled every
single one correctly. If the reader's output were the problem, nobody could have
scored 100% from it. Someone did.

llama3.1's 11 mistakes are of two kinds:

- **8 are the same mistake on clean, whole lines.** `B. Scientific data that will be
  preserved…` is a question, but it is short, bold and lettered — it *looks* like a
  heading, and llama3.1 calls it one. pdfplumber played no part: it handed these
  lines over in one piece.
- **3 are slips on leftover pieces of split answers**, like `about other studies.` —
  the tail of a wrapped sentence. Here pdfplumber created the *difficulty* (a
  fragment that is unreadable alone), but the model committed the *error*: it judged
  the fragment in isolation instead of following the blocks around it. gemma4, given
  the same fragment, kept the label of the sentence it belongs to.

Like a hard exam question: the exam made it hard, the student got it wrong — and
another student answered it correctly.

*(One caution: this verdict is for this document. Sample 6 is different — its
headings are underlined, pdfplumber cannot see underlines, and there the reader
genuinely is at fault.)*
